### Data Preparation

In [3]:
import pandas as pd
import numpy as np

print("Loading reviews dataset...")
reviews_df = pd.read_csv('../../data_resource/reviews.csv')

print(f"Original shape: {reviews_df.shape}")

df = reviews_df[['AuthorId', 'RecipeId', 'Rating']].copy()

df.dropna(inplace=True)

user_counts = df['AuthorId'].value_counts()
active_users = user_counts[user_counts >= 3].index
df = df[df['AuthorId'].isin(active_users)]

recipe_counts = df['RecipeId'].value_counts()
popular_recipes = recipe_counts[recipe_counts >= 3].index
df = df[df['RecipeId'].isin(popular_recipes)]

print(f"Cleaned shape: {df.shape}")
print(f"Unique Users: {df['AuthorId'].nunique()}")
print(f"Unique Recipes: {df['RecipeId'].nunique()}")

Loading reviews dataset...
Original shape: (1401982, 8)
Cleaned shape: (942413, 3)
Unique Users: 44821
Unique Recipes: 98118


### SVD Model Training & Evaluation

In [4]:
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error
from scipy.sparse import csr_matrix
import pickle
import os

print("Loading and cleaning reviews dataset...")
reviews_df = pd.read_csv('../../data_resource/reviews.csv')
df = reviews_df[['AuthorId', 'RecipeId', 'Rating']].dropna()

user_counts = df['AuthorId'].value_counts()
recipe_counts = df['RecipeId'].value_counts()
df = df[df['AuthorId'].isin(user_counts[user_counts >= 3].index)]
df = df[df['RecipeId'].isin(recipe_counts[recipe_counts >= 3].index)]

print("Mapping IDs and Creating Sparse User-Item Matrix...")
user_cat = pd.Categorical(df['AuthorId'])
recipe_cat = pd.Categorical(df['RecipeId'])

user_codes = user_cat.codes
recipe_codes = recipe_cat.codes

user_ids = user_cat.categories.tolist()
recipe_ids = recipe_cat.categories.tolist()

sparse_user_item_matrix = csr_matrix(
    (df['Rating'], (user_codes, recipe_codes)),
    shape=(len(user_ids), len(recipe_ids))
)

print("Training SVD Model (Matrix Factorization)...")
svd = TruncatedSVD(n_components=50, random_state=42)
user_factors = svd.fit_transform(sparse_user_item_matrix)
item_factors = svd.components_

print("Evaluating Model Performance (RMSE)...")
predictions = np.sum(user_factors[user_codes] * item_factors.T[recipe_codes], axis=1)

rmse_score = np.sqrt(mean_squared_error(df['Rating'], predictions))

print(f"\nRMSE Score: {rmse_score:.4f}")
print("NOTE: Please screenshot the RMSE score above as evidence for the 'Advanced suggestions' rubric.\n")

print("Exporting the trained model...")
os.makedirs('../models', exist_ok=True)
model_path = '../models/svd_recommender.pkl'

model_data = {
    'user_factors': user_factors,
    'item_factors': item_factors,
    'user_ids': user_ids,
    'recipe_ids': recipe_ids
}

with open(model_path, 'wb') as f:
    pickle.dump(model_data, f)

print(f"Model successfully saved to {model_path}")

Loading and cleaning reviews dataset...
Mapping IDs and Creating Sparse User-Item Matrix...
Training SVD Model (Matrix Factorization)...
Evaluating Model Performance (RMSE)...

RMSE Score: 4.3164
NOTE: Please screenshot the RMSE score above as evidence for the 'Advanced suggestions' rubric.

Exporting the trained model...
Model successfully saved to ../models/svd_recommender.pkl


### Calculate and Export Global Trending Recipes

In [5]:
import numpy as np
import pickle
import os

print("Calculating global trending recipes based on reviews...")

recipe_stats = reviews_df.groupby('RecipeId').agg(
    review_count=('Rating', 'count'),
    avg_rating=('Rating', 'mean')
).reset_index()

min_reviews = 5
trending_candidates = recipe_stats[recipe_stats['review_count'] >= min_reviews].copy()

trending_candidates['popularity_score'] = trending_candidates['avg_rating'] * np.log1p(trending_candidates['review_count'])

top_trending = trending_candidates.sort_values(by='popularity_score', ascending=False).head(200)

trending_recipe_ids = top_trending['RecipeId'].tolist()

os.makedirs('../models', exist_ok=True)
trending_path = '../models/trending_recipe_ids.pkl'

with open(trending_path, 'wb') as f:
    pickle.dump(trending_recipe_ids, f)

print(f"Successfully extracted {len(trending_recipe_ids)} trending recipes.")
print(f"Exported to {trending_path}")

Calculating global trending recipes based on reviews...
Successfully extracted 200 trending recipes.
Exported to ../models/trending_recipe_ids.pkl
